# Libraries Import

In [1]:
import os, cv2, json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import tensorflow as tf
import tensorflow_hub as hub
from ultralytics import YOLO
# import pytorch as torch``

# from mmengine.config import Config
# from mmengine.registry import MODELS
# from mmengine.runner import load_checkpoint
# from mmaction.apis import init_recognizer

# # This function will now work correctly because we are running from the cloned directory
# from mmaction.utils import register_all_modules
# register_all_modules(init_default_scope=True) # We set the scope manually later

#Colab Base Path
# base_path = "/content/drive/MyDrive/SMT 6/CV/UAS"

#Local Base Path
base_path = ""

#Dataset Paths
data_path = os.path.join(base_path, "match_videos")
# data_path = os.path.join(base_path, "practice_videos")

video_path = os.path.join(data_path, "Knee(Bryan) vs Double(Law) TWT 2024.mp4")
# video_path = os.path.join(data_path, "Knee(Bryan) vs Double(Law) 1 round.mp4")
# video_path = os.path.join(data_path, "Bryan_L.mp4")

# annotation_path = os.path.join(base_path, "match_videos/Knee(Bryan) vs Double(Law) TWT 2024.json")
annotation_path = os.path.join(data_path, "Knee_reindexed.json")

output_dir = os.path.join(data_path, "frames")
kp_dir = os.path.join(data_path)

# video_path = os.path.join(base_path, "Bryan_2/Bryan_15_move_trimmed.mp4")
# annotation_path = os.path.join(base_path, "Bryan_2/Bryan_15_move_2.json")
# output_dir = os.path.join(base_path, "Bryan_2/frames")

# Load movenet
movenet = hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4").signatures['serving_default']
yolo = YOLO("runs/detect/practice/weights/best.pt")
yolo.to("mps")

os.makedirs(output_dir, exist_ok=True)

# movenet multipose bbox and skeleton extraction

In [ ]:
import os, cv2, json
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_hub as hub

# Load movenet
movenet = hub.load("https://tfhub.dev/google/movenet/multipose/lightning/1").signatures['serving_default']
# movenet = hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4").signatures['serving_default']


def run_movenet(image):
    image = tf.image.resize_with_pad(image, 256, 256)
    image = tf.cast(image, dtype=tf.int32)
    image = tf.expand_dims(image, axis=0)

    output = movenet(image)
    # print(image.shape)
    # print(output["output_0"].numpy()[0])
    persons = output['output_0'].numpy()[0]

    keypoints_all = []
    bbox_all = []
    for person in persons:
        # print(person)
        if person[-1] > 0.2:
            keypoints = person[:51].reshape(17, 3)
            bbox = person[51:]
            keypoints_all.append(keypoints)
            bbox_all.append(bbox)
    return keypoints_all, bbox_all

def denormalize_points(points, original_height, original_width, input_size=256):
    """
    Converts normalized keypoints or bounding boxes from MoveNet output
    back to original image coordinates.
    """
    scale = min(input_size / original_height, input_size / original_width)
    new_height = original_height * scale
    new_width = original_width * scale
    pad_y = (input_size - new_height) / 2
    pad_x = (input_size - new_width) / 2

    if len(points) <= 3:
        y, x, c = points
        x_abs = ((x * input_size) - pad_x) / scale
        y_abs = ((y * input_size) - pad_y) / scale
        return (int(y_abs), int(x_abs), c)
    else:
        y_min, x_min, y_max, x_max, c = points
        y_min_abs = ((y_min * input_size) - pad_y) / scale
        y_max_abs = ((y_max * input_size) - pad_y) / scale
        x_min_abs = ((x_min * input_size) - pad_x) / scale
        x_max_abs = ((x_max * input_size) - pad_x) / scale
        return (int(y_min_abs), int(x_min_abs), int(y_max_abs), int(x_max_abs), c)
    
image_path = f"{output_dir}/frame_00191.jpg"
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
original_height, original_width = image.shape[:2]

keypoints_list, bbox_list = run_movenet(image)
# print(keypoints_list)
# print(bbox_list)

for keypoints in keypoints_list:
    for kp in keypoints:
        y, x, c = denormalize_points(kp, original_height, original_width)
        cv2.circle(image, (x, y), 3, (0, 0, 255), thickness=2, lineType=cv2.LINE_AA)

for bboxs in bbox_list:
    for bbox in bbox_list:
        y_min, x_min, y_max, x_max, c = denormalize_points(bbox, original_height, original_width)
        cv2.rectangle(image, (x_min, y_min), (x_max, y_max), (0, 255, 0), thickness=2, lineType=cv2.LINE_AA)

plt.imshow(image)

# Yolo object tracking base

In [ ]:
# YOLO PER IMAGE
def yolo_tracking(image):
    #f"{output_dir}/frame_00191.jpg"
    result = yolo.track(image, tracker="bytetrack.yaml", persist=True, conf=0.5, iou=0.3, classes=[0])[0] 
    # the results of yolo.track contains list of object per frame it tracks, for example image will have 1 object in the list
    # while video will have as many object in it as the video frames
    # we will access the first object as it is an image

    print(result.boxes.xyxy) # print the x1, y1, x2, y2 from boxes of the object
    print(result.boxes.id) # print the id from boxes of the object

    ids = result.boxes.id.numpy().astype(int) # convert boxes ids of selected object to numpy, then to int
    boxes = result.boxes.xyxy.numpy().astype(int) # convert boxes x1, y1, x2, y2 of selected object to numpy, then to int

    id_box_array = np.hstack((ids.reshape(-1, 1), boxes)) 
    # stack ids and boxes horizontally (it only accepts tuple so we encapsulate it with double ()

    print(id_box_array)

def crop_roi(frame, id_box_array):
    id, x1, y1, x2, y2 = id_box_array
    cropped_roi = frame[x1, x2 : y1, y2]
    return cropped_roi

# YOLO PER VIDEO
results = yolo.track(video_path, tracker="bytetrack.yaml", persist=True, conf=0.35, iou=0.3, classes=[0], show=True, save=False)
# results = yolo(video_path, conf=0.5, iou=0.5, classes=[0], show=True, save=False)

# id_box_arrays = np.empty
# for result in results:
#     ids = result.boxes.id.numpy().astype(int)
#     boxes = result.boxes.xyxy.numpy().astype(int)

#     id_box = np.hstack((ids.reshape(-1, 1), boxes))
#     id_box_array = np.concatenate((id_box_array, id_box))

# print(id_box_array)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/34533) /Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/match_videos/Knee(Bryan) vs Double(Law) TWT 2024.mp4: 384x640 2 fighters, 24.0ms
video 1/1 (frame 2/34533) /Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/match_videos/Knee(Bryan) vs Double(Law) TWT 2024.mp4: 384x640 2 fighters, 10.2ms
video 1/1 (frame 3/34533) /Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/match_videos/Knee(Bryan) vs Double(Law) TWT 2024.mp4: 384x640 2 fighters, 9.2ms
video 1/1

NameError: name 'id_box_array' is not defined

: 

# Adjust Annotation

In [ ]:
import json
import os

def adjust_annotations(input_file, output_file=None, frame_adjustment=-1):
    """
    Adjust all frame numbers and recalculate timing data
    
    Args:
        input_file: Path to your current annotations JSON
        output_file: Path for adjusted annotations (if None, creates backup and overwrites)
        frame_adjustment: Number of frames to adjust (negative to reduce, positive to increase)
    """
    
    # Load existing annotations
    with open(input_file, 'r') as f:
        annotations = json.load(f)
    
    # Get FPS from user or estimate (you might want to hardcode this)
    fps = float(input("Enter video FPS (usually 60 for Tekken): ") or "60")
    
    # Create backup if overwriting
    if output_file is None:
        backup_file = input_file.replace('.json', '_backup.json')
        print(f"Creating backup: {backup_file}")
        with open(backup_file, 'w') as f:
            json.dump(annotations, f, indent=2)
        output_file = input_file
    
    adjusted_annotations = {}
    
    for seq_id, seq_data in annotations.items():
        # Adjust frame numbers
        new_start_frame = max(0, seq_data["start_frame"] + frame_adjustment)
        new_end_frame = max(new_start_frame, seq_data["end_frame"] + frame_adjustment)
        
        # Recalculate timing data
        new_start_time = new_start_frame / fps
        new_end_time = new_end_frame / fps
        new_duration_frames = new_end_frame - new_start_frame
        
        # Create adjusted sequence
        adjusted_seq = seq_data.copy()
        adjusted_seq.update({
            "start_frame": new_start_frame,
            "end_frame": new_end_frame,
            "start_time": new_start_time,
            "end_time": new_end_time,
            "duration_frames": new_duration_frames
        })
        
        adjusted_annotations[seq_id] = adjusted_seq
        
        # Print adjustment info
        old_duration = seq_data["duration_frames"]
        print(f"{seq_id}: {old_duration} → {new_duration_frames} frames "
              f"({seq_data['start_frame']}-{seq_data['end_frame']} → {new_start_frame}-{new_end_frame})")
    
    # Save adjusted annotations
    with open(output_file, 'w') as f:
        json.dump(adjusted_annotations, f, indent=2)
    
    print(f"\nAdjustments saved to: {output_file}")
    print(f"Total sequences adjusted: {len(adjusted_annotations)}")

# Usage examples:
if __name__ == "__main__":
    # Adjust your current annotations
    input_file = "match_videos\Knee(Bryan) vs Double(Law) TWT 2024.json"  # Replace with your file path
    
    print("Adjustment options:")
    print("1. Reduce end frames by 1 (end earlier)")
    print("2. Reduce both start and end by 1 (shift entire sequence)")
    print("3. Custom adjustment")
    
    # choice = input("Choose option (1/2/3): ")
    choice = "2"
    
    if choice == "1":
        # Only adjust end frames
        with open(input_file, 'r') as f:
            annotations = json.load(f)
        
        fps = float(input("Enter video FPS: ") or "60")
        
        for seq_id, seq_data in annotations.items():
            seq_data["end_frame"] = max(seq_data["start_frame"], seq_data["end_frame"] - 1)
            seq_data["end_time"] = seq_data["end_frame"] / fps
            seq_data["duration_frames"] = seq_data["end_frame"] - seq_data["start_frame"]
        
        backup_file = input_file.replace('.json', '_backup.json')
        with open(backup_file, 'w') as f:
            json.dump(annotations, f, indent=2)
        
        with open(input_file, 'w') as f:
            json.dump(annotations, f, indent=2)
        
        print(f"End frames reduced by 1. Backup saved as {backup_file}")
    
    elif choice == "2":
        adjust_annotations(input_file, frame_adjustment=-1)
    
    elif choice == "3":
        adjustment = int(input("Enter frame adjustment (negative to reduce): "))
        adjust_annotations(input_file, frame_adjustment=adjustment)

# Recompute Annotation

In [ ]:
import json
import os
import re

def adjust_annotations(input_file, output_file=None, frame_adjustment=0, reindex_only=False,
                       fps=60.0, id_prefix="seq_", preserve_prefix=True, pad_width=3, start_index=1):
    """
    Adjust frames and/or reindex sequence IDs.

    - reindex_only=True: keep frames as-is, renumber keys to contiguous sequence numbers.
      If preserve_prefix=True the function keeps the original prefix (e.g. "p1_sequence_")
      and only replaces the numeric suffix.
    - pad_width controls zero-padding of the new numeric suffix.
    - start_index allows starting numbering at a given integer.
    """
    with open(input_file, 'r', encoding='utf-8') as f:
        annotations = json.load(f)

    if output_file is None:
        backup_file = input_file.replace('.json', '_backup.json')
        print(f"Creating backup: {backup_file}")
        with open(backup_file, 'w', encoding='utf-8') as f:
            json.dump(annotations, f, indent=2, ensure_ascii=False)
        output_file = input_file

    if reindex_only:
        items = sorted(annotations.items(), key=lambda kv: int(kv[1].get("start_frame", 0)))
        adjusted_annotations = {}
        mapping = {}

        for idx, (old_id, seq_data) in enumerate(items, start=start_index):
            # determine prefix to preserve (e.g. "p1_sequence_"); fallback to id_prefix
            if preserve_prefix:
                m = re.match(r'^(.*?_sequence_)', old_id)
                if m:
                    prefix = m.group(1)
                else:
                    parts = old_id.rsplit('_', 1)
                    prefix = (parts[0] + '_') if len(parts) > 1 else id_prefix
            else:
                prefix = id_prefix

            # create numeric suffix with optional zero-padding
            if pad_width and int(pad_width) > 0:
                new_num = f"{idx:0{pad_width}d}"
            else:
                new_num = str(idx)

            new_id = f"{prefix}{new_num}"

            new_seq = seq_data.copy()
            start_f = int(new_seq.get("start_frame", 0))
            end_f = int(new_seq.get("end_frame", 0))
            if end_f < start_f:
                end_f = start_f

            new_seq["start_frame"] = start_f
            new_seq["end_frame"] = end_f
            new_seq["duration_frames"] = end_f - start_f
            new_seq["start_time"] = start_f / float(fps)
            new_seq["end_time"] = end_f / float(fps)

            adjusted_annotations[new_id] = new_seq
            mapping[old_id] = new_id

        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(adjusted_annotations, f, indent=2, ensure_ascii=False)

        print(f"Reindexed {len(mapping)} sequences. Mapping (old -> new):")
        for old, new in mapping.items():
            print(f"  {old} -> {new}")
        print(f"Saved reindexed annotations to: {output_file}")
        return

    # fallback: perform frame shift + recompute (unchanged)
    adjusted_annotations = {}
    for seq_id, seq_data in annotations.items():
        new_start_frame = max(0, int(seq_data.get("start_frame", 0)) + int(frame_adjustment))
        new_end_frame = max(new_start_frame, int(seq_data.get("end_frame", 0)) + int(frame_adjustment))

        new_start_time = new_start_frame / float(fps)
        new_end_time = new_end_frame / float(fps)
        new_duration_frames = new_end_frame - new_start_frame

        adjusted_seq = seq_data.copy()
        adjusted_seq.update({
            "start_frame": new_start_frame,
            "end_frame": new_end_frame,
            "start_time": new_start_time,
            "end_time": new_end_time,
            "duration_frames": new_duration_frames
        })

        adjusted_annotations[seq_id] = adjusted_seq

        old_duration = seq_data.get("duration_frames", new_duration_frames)
        print(f"{seq_id}: {old_duration} → {new_duration_frames} frames "
              f"({seq_data.get('start_frame')}-{seq_data.get('end_frame')} → {new_start_frame}-{new_end_frame})")

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(adjusted_annotations, f, indent=2, ensure_ascii=False)

    print(f"\nAdjustments saved to: {output_file}")
    print(f"Total sequences adjusted: {len(adjusted_annotations)}")

adjust_annotations(
  input_file="match_videos\\Knee(Bryan) vs Double(Law) TWT 2024.json",
  output_file="match_videos\\Knee_reindexed.json",
  reindex_only=True,
  fps=60.0,
  preserve_prefix=True,
  pad_width=0,
  start_index=1
)

# Export Frame to JPG CV2

In [ ]:
# Open and load annotations json
with open(annotation_path, "r") as f:
    annotations = json.load(f)

# Read video
cap = cv2.VideoCapture(video_path)
if not cap.isOpened:
    print("Error, could not open video")

else: # cut videos to chunks of sequences consists of move frames
    for sequences_id, info in annotations.items():
        print(sequences_id, info)
        for frame_idx in range(info["start_frame"], info["end_frame"] + 1):
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if ret:
                cv2.imwrite(os.path.join(output_dir, f"frame_{frame_idx:05d}.jpg"), frame)

cap.release()


# Export Frame to long mp4 CV2

In [3]:
# Export concatenated video at slower playback speed (set slow_factor = 2,3,4 for half/third/quarter)
with open(annotation_path, "r") as f:
    annotations = json.load(f)

items = sorted(annotations.items(), key=lambda kv: int(kv[1].get("start_frame", 0)))

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Error: could not open video")
else:
    os.makedirs(output_dir, exist_ok=True)
    # find first readable frame to get fps/size
    first_ok = False
    for _, info in items:
        start = int(info["start_frame"])
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
        ret, frame = cap.read()
        if ret:
            height, width = frame.shape[:2]
            fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
            first_ok = True
            break
    if not first_ok:
        print("No readable frames found in annotated ranges.")
    else:
        slow_factor = 4   # set to 2 (half speed), 3 (third speed), or 4 (quarter speed)
        method = "fps"    # "fps" to lower output fps, or "duplicate" to duplicate each frame int(slow_factor) times
        out_path = os.path.join(output_dir, f"all_sequences_slow_{slow_factor}x.mp4")

        # preferred: lower output fps (works for most cases, fractional fps allowed)
        target_fps = float(fps) / float(slow_factor)

        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(out_path, fourcc, target_fps, (width, height))

        if method == "duplicate":
            dup = max(1, int(round(slow_factor)))  # integer duplication factor

        black_frame = np.zeros((height, width, 3), dtype=np.uint8)
        separator_seconds = 0.5
        sep_frames = int(round(separator_seconds * target_fps))

        for seq_id, info in items:
            start = int(info["start_frame"])
            end = int(info["end_frame"])
            if end < start:
                print(f"Skipping {seq_id}: end < start")
                continue

            for fidx in range(start, end + 1):
                cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
                ret, frame = cap.read()
                if not ret:
                    print(f"Missing frame {fidx} for {seq_id}, skipping")
                    continue

                # small overlay (optional)
                label = str(info.get("move", info.get("label", "")))
                seq_text = f"{seq_id}: {label}"
                cv2.putText(frame, seq_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2, cv2.LINE_AA)

                if method == "fps":
                    writer.write(frame)               # writing at lower target_fps slows playback
                else:
                    for _ in range(dup):              # duplicate frames to slow (keeps integer fps)
                        writer.write(frame)

            for _ in range(sep_frames):
                writer.write(black_frame)

        writer.release()
        print(f"Saved slowed video: {out_path} (slow_factor={slow_factor}, method={method})")

cap.release()

Saved slowed video: match_videos/frames\all_sequences_slow_4x.mp4 (slow_factor=4, method=fps)


# Export Frame to long mp4 FFMPEG

In [3]:
import subprocess, os, json, tempfile, pathlib

slow_factor = 4.0  # 2.0, 3.0, 4.0
ffmpeg = "ffmpeg"
font = "C\\:/Windows/Fonts/arial.ttf"

with open(annotation_path, "r", encoding="utf-8") as f:
    annotations = json.load(f)

items = sorted(annotations.items(), key=lambda kv: int(kv[1].get("start_frame", 0)))
if not items:
    raise ValueError("No annotations found.")

def _escape_drawtext(s: str) -> str:
    # escape characters that drawtext treats specially
    s = s.replace("\\", "\\\\")   # backslash first
    s = s.replace(":", "\\:")
    s = s.replace("'", "\\'")
    s = s.replace("%", "\\%")
    s = s.replace("\n", " ")
    s = s.replace("\r", " ")
    return s

parts = []
concat_inputs = []
valid = 0

for _, (seq_id, info) in enumerate(items):
    start = int(info["start_frame"])
    end = int(info["end_frame"])
    if end < start:
        continue

    label = str(info.get("move", info.get("label", "")))
    text = _escape_drawtext(f"{seq_id} {label}".strip())

    parts.append(
        f"[0:v]"
        f"select='between(n\\,{start}\\,{end})',"
        f"setpts=PTS-STARTPTS,"
        f"setpts={slow_factor}*PTS,"
        f"drawtext=fontfile='{font}':text='{text}':x=10:y=20:fontsize=24:"
        f"fontcolor=white:box=1:boxcolor=0x00000099:boxborderw=6"
        f"[v{valid}]"
    )
    concat_inputs.append(f"[v{valid}]")
    valid += 1

if valid == 0:
    raise ValueError("No valid (start_frame,end_frame) ranges found.")

filter_complex = ";".join(parts) + ";" + "".join(concat_inputs) + f"concat=n={valid}:v=1:a=0[outv]"

out_final = os.path.join(output_dir, f"all_sequences_slow_{int(slow_factor)}x_ffmpeg_overlay.mp4")

# Write filtergraph to temp file to avoid Windows cmdline length limit
tmpdir = pathlib.Path(tempfile.mkdtemp())
fg_path = tmpdir / "filtergraph.txt"
fg_path.write_text(filter_complex, encoding="utf-8")

cmd = [
    ffmpeg, "-y",
    "-i", video_path,
    "-filter_complex_script", str(fg_path),
    "-map", "[outv]",
    "-an",
    "-c:v", "libx264",
    "-preset", "veryfast",
    "-crf", "20",
    out_final
]

print(f"Running ffmpeg (single pass) with overlay for {valid} segments...")
print("Filtergraph file:", fg_path)

try:
    subprocess.run(cmd, check=True)
except subprocess.CalledProcessError as e:
    print("ffmpeg failed")
    print("command:", " ".join(cmd))
    raise

print("Saved:", out_final)

Running ffmpeg (single pass) with overlay for 153 segments...
Filtergraph file: C:\Users\MYBOOK~1\AppData\Local\Temp\tmpiq3wfygf\filtergraph.txt
Saved: match_videos/frames\all_sequences_slow_4x_ffmpeg_overlay.mp4


# Create Fighter Dataset

In [ ]:
input_size = 192
cap = cv2.VideoCapture(video_path)
print(video_path, os.path.exists(video_path))

frame_count = 0

frame_dir = os.path.join(base_path, "fighter_dataset/images")
label_dir = os.path.join(base_path, "fighter_dataset/labels")

os.makedirs(frame_dir, exist_ok=True)
os.makedirs(label_dir, exist_ok=True)

def yolo_detect(image, min_w = 200, min_h = 200):
    # Depending on how you fixed the initialization earlier, 
    # you might have device='mps' here, which is perfectly fine!
    result = yolo(image, conf=0.3, iou=0.5, classes=[0])[0]

    if result.boxes is None or len(result.boxes) == 0:
        return []
    
    # 🎯 FIX HERE: Add .cpu() before .numpy()
    boxes = result.boxes.xyxy.cpu().numpy().astype(int) 

    filtered_boxes = []
    for box in boxes:
        x1, y1, x2, y2 = box
        w = x2 - x1
        h = y2 - y1
        if w > min_w and h > min_h: 
            filtered_boxes.append(box)

    return np.array(filtered_boxes)

def pad_box(h, w, box, padding=25):
    x1, y1, x2, y2 = box
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(w, x2 + padding)
    y2 = min(h, y2 + padding)

    return np.array([x1, y1, x2, y2], dtype=int)

def xyxy_to_yolo_normalized(box, img_width, img_height):
    x1, y1, x2, y2 = box
    
    # Calculate width and height
    w = x2 - x1
    h = y2 - y1
    
    # Calculate center x and center y
    x_center = x1 + (w / 2)
    y_center = y1 + (h / 2)
    
    # Normalize by image dimensions
    x_center_n = x_center / img_width
    y_center_n = y_center / img_height
    w_n = w / img_width
    h_n = h / img_height
    
    return [x_center_n, y_center_n, w_n, h_n]

# Get total frames to avoid reading past the end
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_step = 10

for frame_count in range(0, total_frames, frame_step):
    # Jump directly to the desired frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_count)
    ret, frame_bgr = cap.read()
    
    if not ret:
        print("End of video or read error")
        break

    if frame_count == 0:
        original_height, original_width = frame_bgr.shape[:2]
        print("Original height, original_width", original_height, original_width)

    # Detect objects
    box_array = yolo_detect(frame_bgr) 
    
    if box_array is not None and len(box_array) > 0:
        # --- 1. SAVE THE IMAGE ---
        base_filename = f"frame_{frame_count:05d}"
        image_path = os.path.join(frame_dir, f"{base_filename}.jpg")
        cv2.imwrite(image_path, frame_bgr)

        # Apply padding to the detected boxes
        padded_boxes = np.array(
            [pad_box(original_height, original_width, box) for box in box_array],
            dtype=int 
        )
        
        # --- 2. SAVE THE LABELS (.txt) ---
        label_path = os.path.join(label_dir, f"{base_filename}.txt")
        
        with open(label_path, "w") as f:
            for box in padded_boxes:
                norm_box = xyxy_to_yolo_normalized(box, original_width, original_height)
                f.write(f"0 {norm_box[0]:.6f} {norm_box[1]:.6f} {norm_box[2]:.6f} {norm_box[3]:.6f}\n")

cap.release()
cv2.destroyAllWindows() # Clean up windows

# Train Val Split

In [1]:
import os
import shutil
import random
import yaml

# Adjust base_path if needed
base_path = ""
dataset_dir = os.path.join(base_path, "fighter_dataset")  # Change to your dataset directory

# Source directories from your previous step
src_images_dir = os.path.join(dataset_dir, "images")
src_labels_dir = os.path.join(dataset_dir, "labels") # Note: YOLO prefers 'labels' plural, we will fix this below

# YOLO standard split directories
train_images_dir = os.path.join(dataset_dir, "images", "train")
val_images_dir = os.path.join(dataset_dir, "images", "val")
train_labels_dir = os.path.join(dataset_dir, "labels", "train")
val_labels_dir = os.path.join(dataset_dir, "labels", "val")

# Ensure target directories exist
for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
    os.makedirs(dir_path, exist_ok=True)

# Get all image files and shuffle them for a random split
image_files = [f for f in os.listdir(src_images_dir) if f.endswith('.jpg')]
random.shuffle(image_files)

# Define split ratio (80% Train, 20% Val)
split_ratio = 0.8
split_index = int(len(image_files) * split_ratio)

train_files = image_files[:split_index]
val_files = image_files[split_index:]

def populate_split(file_list, dest_img_dir, dest_lbl_dir):
    for img_file in file_list:
        # Source paths
        src_img = os.path.join(src_images_dir, img_file)
        
        lbl_file = img_file.replace('.jpg', '.txt')
        src_lbl = os.path.join(src_labels_dir, lbl_file)
        
        # Destination paths
        dest_img = os.path.join(dest_img_dir, img_file)
        dest_lbl = os.path.join(dest_lbl_dir, lbl_file)
        
        # Move files
        if os.path.exists(src_img):
            shutil.move(src_img, dest_img)
        if os.path.exists(src_lbl):
            shutil.move(src_lbl, dest_lbl)

# Execute the split
populate_split(train_files, train_images_dir, train_labels_dir)
populate_split(val_files, val_images_dir, val_labels_dir)

# Clean up the old, now empty 'label' directory if you want
if os.path.exists(src_labels_dir) and not os.listdir(src_labels_dir):
    os.rmdir(src_labels_dir)

# Generate the data.yaml file required by YOLO
yaml_content = {
    'path': os.path.abspath(dataset_dir),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': {0: 'fighter'}
}

yaml_path = os.path.join(dataset_dir, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"✅ Split Complete! {len(train_files)} Training | {len(val_files)} Validation")
print(f"✅ data.yaml generated at: {yaml_path}")

✅ Split Complete! 612 Training | 154 Validation
✅ data.yaml generated at: fighter_dataset/data.yaml


# Yolo Training

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
model.train(
    data='fighter_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device='mps',
    patience=50,
    workers=4,
    half=True,
    seed=42,
)

# Review KP

In [3]:
# Load the saved data
p1_path = os.path.join(kp_dir, "player1_kp.npy")
p1_data = np.load(p1_path)

print(f"--- Data Diagnostics ---")
print(f"Shape: {p1_data.shape} (Expected: [Frames, 17, 3])")

# Check Frame 0 specifically (since that's where your image shows the issue)
frame0 = p1_data[0]
print(f"\nFrame 0 Sample (Nose [y, x, conf]): {frame0[0]}")

# Check for NaNs
print(f"Has NaNs: {np.isnan(p1_data).any()}")

# Check Value Range (To determine if it's Normalized 0-1 or Pixels)
y_coords = p1_data[:, :, 0]
x_coords = p1_data[:, :, 1]
print(f"\nY Range: Min={np.nanmin(y_coords):.4f}, Max={np.nanmax(y_coords):.4f}")
print(f"X Range: Min={np.nanmin(x_coords):.4f}, Max={np.nanmax(x_coords):.4f}")

if np.nanmax(y_coords) > 1.5 or np.nanmax(x_coords) > 1.5:
    print("\n⚠️ DIAGNOSIS: Data is likely stored in PIXELS (Denormalized).")
    print("ACTION: Remove the 'x * width' and 'y * height' multiplication in the viz script.")
else:
    print("\n✅ DIAGNOSIS: Data is likely NORMALIZED (0.0 - 1.0).")
    print("ACTION: Ensure width/height in viz script match the video source exactly.")

--- Data Diagnostics ---
Shape: (306, 17, 3) (Expected: [Frames, 17, 3])

Frame 0 Sample (Nose [y, x, conf]): [    0.27083      0.3125     0.65366]
Has NaNs: False

Y Range: Min=0.2319, Max=0.9264
X Range: Min=0.1602, Max=0.5359

✅ DIAGNOSIS: Data is likely NORMALIZED (0.0 - 1.0).
ACTION: Ensure width/height in viz script match the video source exactly.


# Draw kp on vid

In [4]:
import os
import cv2
import numpy as np

# MoveNet 17-keypoint skeleton edges
# (start_idx, end_idx)
MOVENET_EDGES = [
    (0, 1), (0, 2), (1, 3), (2, 4),          # head
    (5, 6),                                  # shoulders
    (5, 7), (7, 9),                          # left arm
    (6, 8), (8, 10),                         # right arm  <-- add these
    (5, 11), (6, 12), (11, 12),              # torso
    (11, 13), (13, 15),                      # left leg
    (12, 14), (14, 16)                       # right leg
]

def _to_pixel_xy(kp, width, height, is_normalized):
    y, x, conf = kp
    if np.isnan(y) or np.isnan(x):
        return None
    if is_normalized:
        px = int(x * width)
        py = int(y * height)
    else:
        px = int(x)
        py = int(y)
    # clamp to frame
    px = max(0, min(width - 1, px))
    py = max(0, min(height - 1, py))
    return (px, py, conf)

def visualize_robust(video_input_path, output_path, p1_kp_path, p2_kp_path, conf_th=0.2):
    p1_kp = np.load(p1_kp_path)
    p2_kp = np.load(p2_kp_path)

    cap = cv2.VideoCapture(video_input_path)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0

    print(f"Video Resolution: {width}x{height}")

    is_normalized = np.nanmax(p1_kp[:, :, 0]) <= 1.5
    print(f"Data Detected as: {'NORMALIZED (0-1)' if is_normalized else 'PIXELS'}")

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    n_frames = min(len(p1_kp), len(p2_kp))
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_idx >= n_frames:
            break

        for kp_array, color in [(p1_kp[frame_idx], (0, 255, 0)), (p2_kp[frame_idx], (255, 0, 0))]:
            pts = [_to_pixel_xy(kp, width, height, is_normalized) for kp in kp_array]

            # Draw limbs (lines)
            for i, j in MOVENET_EDGES:
                pi = pts[i]
                pj = pts[j]
                if pi is None or pj is None:
                    continue
                if pi[2] < conf_th or pj[2] < conf_th:
                    continue
                cv2.line(frame, (pi[0], pi[1]), (pj[0], pj[1]), color, 2, cv2.LINE_AA)

            # Draw joints (circles)
            for p in pts:
                if p is None:
                    continue
                if p[2] < conf_th:
                    continue
                cv2.circle(frame, (p[0], p[1]), 4, color, -1, cv2.LINE_AA)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    print("Done.")

# --- Execution ---
# Update these paths to match your directory structure
p1_path = os.path.join(kp_dir, "player1_kp.npy")
p2_path = os.path.join(kp_dir, "player2_kp.npy")
output_vid = os.path.join(kp_dir, "kp_visualization.mp4")

# Run this
visualize_robust(video_path, output_vid, p1_path, p2_path)

Video Resolution: 1352x878
Data Detected as: NORMALIZED (0-1)
Done.
